# הפה של נוגה – סנכרון שפתיים עם LatentSync (חינם, על כרטיס מסך של גוגל)

**לפני הכל:** בתפריט למעלה: **Runtime → Change runtime type → T4 GPU → Save**.

אחר כך מריצים את התאים לפי הסדר (לוחצים על ▶ ליד כל תא, או Ctrl+Enter):
1. **בדיקת כרטיס מסך** – צריך לראות Tesla T4.
2. **התקנה** – כ-10 דקות. אזהרות אדומות זה בסדר; רק שגיאה בסוף התא היא בעיה.
3. **חיבור ל-Google Drive** – גוגל תבקש אישור; מאשרים.
4. **בדיקה קצרה (15 שניות)** – התוצאה נשמרת ב-Drive בתיקייה `noga_lipsync`. שולחים לClaude הודעה כשזה מוכן.
5. **הרצה מלאה** – רק אחרי שClaude אישר את הבדיקה. אפשר להפסיק ולהמשיך מחר; מה שכבר נוצר לא ייווצר שוב.

אם יש לכם סרטון בסיס חדש של נוגה מ-Gemini (פה סגור), שימו אותו ב-Drive בשם `noga_lipsync/base.mp4` לפני שלב 4.

In [ ]:
# 1) בדיקת כרטיס מסך
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'אין כרטיס מסך: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
%%bash
# 2) התקנה (כ-10 דקות)
set -e
cd /content
if [ ! -d LatentSync ]; then git clone -q https://github.com/bytedance/LatentSync.git; fi
cd LatentSync && git checkout -q 7b380d6      # LatentSync 1.5 (fits the free T4)
pip -q install uv
uv python install 3.10 >/dev/null
[ -d /content/ls310 ] || uv venv -q --python 3.10 /content/ls310
export VIRTUAL_ENV=/content/ls310
uv pip install -q --index-strategy unsafe-best-match -r requirements.txt
# T4 handles half precision well; the original check only allows newer cards
sed -i 's/get_device_capability()\[0\] > 7/get_device_capability()[0] >= 7/' scripts/inference.py
/content/ls310/bin/python - <<'EOF'
from huggingface_hub import hf_hub_download
for f in ['whisper/tiny.pt', 'latentsync_unet.pt']:
    hf_hub_download('ByteDance/LatentSync-1.5', f, local_dir='checkpoints')
print('checkpoints ok')
EOF
echo 'ההתקנה הסתיימה בהצלחה'

In [ ]:
# 3) חיבור ל-Google Drive + הכנת סרטון הבסיס
from google.colab import drive
import os, subprocess, json, urllib.request
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/noga_lipsync'; os.makedirs(OUT, exist_ok=True)
BASE_URL = 'https://ransimhi111-commits.github.io/noga-play/lab/'
os.makedirs('/content/in', exist_ok=True)
def get(name, dst):
    if not os.path.exists(dst): urllib.request.urlretrieve(BASE_URL + name, dst)
    return dst
src = os.path.join(OUT, 'base.mp4')
if os.path.exists(src):
    w = int(subprocess.run(['ffprobe','-v','error','-select_streams','v:0','-show_entries','stream=width','-of','csv=p=0',src],capture_output=True,text=True).stdout.strip())
    vf = 'crop=450:540:414:10' if w == 1280 else 'scale=450:540'   # same framing as the app
    subprocess.run(['ffmpeg','-v','error','-y','-i',src,'-an','-vf',vf,'-r','25','-crf','16','/content/in/base.mp4'],check=True)
    print('משתמש בסרטון הבסיס החדש מה-Drive')
else:
    subprocess.run(['ffmpeg','-v','error','-y','-i',get('noga_base.mp4','/content/in/noga_base.mp4'),'-an','-r','25','-crf','16','/content/in/base.mp4'],check=True)
    print('משתמש בסרטון הבסיס הנוכחי של נוגה')
def lipsync(audio, out):
    r = subprocess.run(['/content/ls310/bin/python','-m','scripts.inference','--unet_config_path','configs/unet/stage2.yaml',
        '--inference_ckpt_path','checkpoints/latentsync_unet.pt','--inference_steps','20','--guidance_scale','1.5',
        '--video_path','/content/in/base.mp4','--audio_path',audio,'--video_out_path',out], cwd='/content/LatentSync', capture_output=True, text=True)
    if r.returncode != 0 or not os.path.exists(out):
        print(r.stdout[-3000:]); print(r.stderr[-3000:]); raise RuntimeError('LatentSync נכשל – שלחו לClaude צילום של ההודעה הזו')
print('מוכן')

In [ ]:
# 4) בדיקה קצרה: 3 משפטים של נוגה (כ-3–5 דקות)
import time, shutil
from IPython.display import HTML
from base64 import b64encode
t = time.time()
lipsync(get('noga_test_audio.wav', '/content/in/test.wav'), '/content/test_latentsync.mp4')
shutil.copy('/content/test_latentsync.mp4', os.path.join(OUT, 'test_latentsync.mp4'))
print('הבדיקה מוכנה ונשמרה ב-Drive: noga_lipsync/test_latentsync.mp4 (%.0f שניות)' % (time.time() - t))
HTML('<video width=360 controls src="data:video/mp4;base64,%s"></video>' % b64encode(open('/content/test_latentsync.mp4','rb').read()).decode())

In [ ]:
# 5) הרצה מלאה – רק אחרי אישור של Claude. אפשר לעצור (■) ולהמשיך מאוחר יותר.
import time
man = json.load(urllib.request.urlopen(BASE_URL + 'manifest.json'))
names = sorted(man)
for i, name in enumerate(names):
    dst = os.path.join(OUT, name + '.mp4')
    if os.path.exists(dst): print(name, 'כבר קיים'); continue
    t = time.time()
    tmp = '/content/' + name + '.mp4'
    lipsync(get('chunks/' + name + '.wav', '/content/in/' + name + '.wav'), tmp)
    shutil.copy(tmp, dst)
    print('%s מוכן (%d/%d, %.0f דקות)' % (name, i + 1, len(names), (time.time() - t) / 60))
print('הכל מוכן! תגידו לClaude שאפשר להמשיך.')